# CS 3110/5110: Data Privacy
## In-Class Exercise, week of 11/03/2025

In [55]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)

def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)

def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [v + np.random.normal(loc=0, scale=sensitivity * np.sqrt(2*np.log(1.25/delta)) / epsilon)
            for v in vec]

def pct_error(orig, priv):
    return np.abs(orig - priv)/orig * 100.0

adult = pd.read_csv('https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv')

## Question 1

Implement an *encoding function* for randomized response to a "yes/no" question.

Reference [Chapter 13](https://uvm-plaid.github.io/programming-dp/notebooks/ch13.html#randomized-response).

In [56]:
def encode_rand_resp_yes_no(true_response):
    # fip a coin
    c1 = np.random.randint(0,2)
    # heads -> answer truthfully
    if c1 == 0:
        return true_response
    # heads -> flip coin
    else:
        c2 = np.random.randint(0,2)
    #           heads -> yes
        if c2 == 0:
            return True
    #           tails -> no
        else:
            return False


for _ in range(5):
    print('Randomized response:', encode_rand_resp_yes_no(True))

Randomized response: True
Randomized response: True
Randomized response: True
Randomized response: True
Randomized response: True


In [57]:
# TEST CASE
assert np.sum([encode_rand_resp_yes_no(True) for _ in range(100)]) > 60
assert np.sum([encode_rand_resp_yes_no(True) for _ in range(100)]) < 90

In [58]:
#what goes to db
noisy_db = (adult['Education'] == 'Bachelors').apply(encode_rand_resp_yes_no)
noisy_db

0         True
1         True
2        False
3         True
4         True
         ...  
32556    False
32557    False
32558    False
32559    False
32560    False
Name: Education, Length: 32561, dtype: bool

In [59]:
len(adult[adult['Education'] == 'Bachelors'])


5355

In [60]:
len(noisy_db[noisy_db])

10674

## Question 2

Implement a *decoding function* for randomized response to a "yes/no" question.

In [61]:
def decode_rand_resp_yes_no(responses):
    all_yesses = np.sum(responses)
    fake_yesses = .25 * len(responses)
    true_yesses = all_yesses - fake_yesses
    approx_true = true_yesses * 2
    return approx_true

# Example: 1000 "yesses" and 500 "nos"
true_responses = [True for _ in range(1000)] + [False for _ in range(500)]
print('Number of "True" yesses:', np.sum(true_responses))

# Randomized responses
# Each response satisfies 1.09-differential privacy
rand_responses = [encode_rand_resp_yes_no(r) for r in true_responses]

# Decode the responses by subtracting "fake" yesses
print('Decoded randomized response yesses:', decode_rand_resp_yes_no(rand_responses))

Number of "True" yesses: 1000
Decoded randomized response yesses: 1046.0


In [62]:
# TEST CASE
true_responses = [True for _ in range(1000)] + [False for _ in range(500)]

# Randomized responses
# Each response satisfies 1.09-differential privacy
rand_responses = [encode_rand_resp_yes_no(r) for r in true_responses]

# Decode the responses by subtracting "fake" yesses
assert decode_rand_resp_yes_no(rand_responses) < 1100
assert decode_rand_resp_yes_no(rand_responses) > 900

## Question 3

Use the definition of randomized response above to answer the question:

*How many individuals in the `adult` dataset have `Occupation` = `Sales`?*

In [63]:
# take a single occupation from the adult dataset, and return a single response
def encode_response_sales(response):
    return encode_rand_resp_yes_no(response == "Sales")
    
def decode_responses_sales(responses):
    return decode_rand_resp_yes_no(responses)

responses = [encode_response_sales(r) for r in adult['Occupation']]
decode_responses_sales(responses)

np.float64(3851.5)

In [64]:
# How accurate is the answer above?
true_sales = np.sum(adult['Occupation'] == 'Sales')
print('True number of salespeople:', true_sales)

True number of salespeople: 3650


## Question 4

Implement the *encode* and *perturb* steps for Optimized Unary Hashing.

In [65]:
domain = adult['Occupation'].dropna().unique()
domain

def encode(response):
    encoded = [0 for _ in domain]
    idx = list(domain).index(response)
    encoded[idx] = 1
    return encoded

print(encode('Sales'))

def petrub_bit(bit):
    p = .95
    q = .05
    
    sample = np.random.random()

    if bit == 1:
        if sample <= p :
            return 1
        else:
            return 0
    elif bit == 0:
        if sample <= q:
            return 1
        else:
            return 0


def perturb(encoded_response):
    return [petrub_bit(b) for b in encoded_response]

perturb(encode('Sales'))

[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]


[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]

In [66]:
# As in randomized response, flipping of bits causes issues in the answers
# This is the perturbed answer (without decoding)
counts = np.sum([perturb(encode(r)) for r in adult['Occupation'].dropna()], axis=0)
list(zip(domain, counts))

[('Adm-clerical', np.int64(4891)),
 ('Exec-managerial', np.int64(5117)),
 ('Handlers-cleaners', np.int64(2784)),
 ('Prof-specialty', np.int64(5247)),
 ('Other-service', np.int64(4437)),
 ('Sales', np.int64(4776)),
 ('Craft-repair', np.int64(5210)),
 ('Transport-moving', np.int64(3001)),
 ('Farming-fishing', np.int64(2479)),
 ('Machine-op-inspct', np.int64(3326)),
 ('Tech-support', np.int64(2387)),
 ('Protective-serv', np.int64(2086)),
 ('Armed-Forces', np.int64(1553)),
 ('Priv-house-serv', np.int64(1606))]

In [67]:
# This is the real answer
counts = np.sum([encode(r) for r in adult['Occupation'].dropna()], axis=0)
list(zip(domain, counts))

[('Adm-clerical', np.int64(3770)),
 ('Exec-managerial', np.int64(4066)),
 ('Handlers-cleaners', np.int64(1370)),
 ('Prof-specialty', np.int64(4140)),
 ('Other-service', np.int64(3295)),
 ('Sales', np.int64(3650)),
 ('Craft-repair', np.int64(4099)),
 ('Transport-moving', np.int64(1597)),
 ('Farming-fishing', np.int64(994)),
 ('Machine-op-inspct', np.int64(2002)),
 ('Tech-support', np.int64(928)),
 ('Protective-serv', np.int64(649)),
 ('Armed-Forces', np.int64(9)),
 ('Priv-house-serv', np.int64(149))]

## Question 5

Implement the *aggregate* step for Optimized Unary Hashing.

In [68]:
def aggregate(responses):
    p = .7
    q = .25

    sums = np.sum(responses, axis=0)
    n = len(responses)
    return [(v-n*q)/(p-q) for v in sums]

responses = [perturb(encode(r)) for r in adult['Occupation'].dropna()]
counts = aggregate(responses)
list(zip(domain, counts))

[('Adm-clerical', np.float64(-6074.444444444445)),
 ('Exec-managerial', np.float64(-5490.000000000001)),
 ('Handlers-cleaners', np.float64(-10990.000000000002)),
 ('Prof-specialty', np.float64(-5470.000000000001)),
 ('Other-service', np.float64(-7021.111111111112)),
 ('Sales', np.float64(-6321.111111111111)),
 ('Craft-repair', np.float64(-5570.000000000001)),
 ('Transport-moving', np.float64(-10603.333333333334)),
 ('Farming-fishing', np.float64(-11583.333333333334)),
 ('Machine-op-inspct', np.float64(-9803.333333333334)),
 ('Tech-support', np.float64(-11810.000000000002)),
 ('Protective-serv', np.float64(-12387.77777777778)),
 ('Armed-Forces', np.float64(-13801.111111111113)),
 ('Priv-house-serv', np.float64(-13192.222222222223))]

In [69]:
def unary_epsilon(p,q):
    return np.log((p*(1-q))) / ((1-p)*q)

In [70]:
#smaller p -> smaller epsilon
# original was 1 -> w prop. p we keep it a 1
# prob. of false negative is 1-p
# prob of true positive is p

#smaller q -> larger epsilon
# original was 0 -> w prop. p we keep it a 1
# prob. of false negative is 1
# prob of true positive is 1-q



unary_epsilon(p=.3,q=.15)

np.float64(-13.014206988797246)